# Final Model

In this notebook, we evaluate the performance of one of our chosen models. 

We compare it's performance against the baseline and the more heavily feature engineered model.

In [1]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score, recall_score, 
                             f1_score, fbeta_score, confusion_matrix, log_loss, 
                             classification_report, average_precision_score, precision_recall_curve)

import warnings
warnings.filterwarnings("ignore")

pd.set_option('display.max_colwidth', None)

In [2]:
Xt = pd.read_csv("../../data/cleaned_data/train.csv")
Xv = pd.read_csv("../../data/cleaned_data/test.csv")

yt = Xt["target"]
Xt.drop(columns=["target"],inplace=True)
yv = Xv["target"]
Xv.drop(columns=["target"],inplace=True)
Xt.drop(columns=["patient_nbr"],inplace=True)
Xv.drop(columns=["patient_nbr"],inplace=True)

In [3]:
chosen_features = ["number_inpatient",
            "number_emergency", "payer_code",
            "discharge_disposition_id",
            "diag_1", "number_diagnoses",
            "num_medications"]

Xt = Xt[chosen_features]
Xv = Xv[chosen_features]

In [4]:
num_cols = Xt.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = Xt.select_dtypes(include=['object','category']).columns.tolist()
features = num_cols+cat_cols

In [5]:
num = Pipeline([
    # ("eng", FunctionTransformer(engineer)),
    ("imp", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

cat = Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Missing")), 
                ("oh", OneHotEncoder(handle_unknown="infrequent_if_exist",min_frequency=100))])


prep = ColumnTransformer([("num", num, num_cols),
                          ("cat", cat, cat_cols)])

clf = Pipeline([("prep", prep), 
                ("model", LogisticRegression(class_weight={0: 1, 1: 22}, 
                                             max_iter=1_000,
                                             C=0.2, 
                                             random_state=42)
                                             )])

In [6]:
# Fit the model to the training data
clf.fit(Xt,yt)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('prep', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains s

In [7]:
probs = clf.predict_proba(Xv)[:,1]
y_pred=None 
proba=probs
threshold=0.65989


# if y_pred is not provided but thresholds and the probabilities are, define predictions
if threshold is not None and proba is not None and y_pred is None:
    y_pred = (proba >= threshold).astype(int)

# obtain various metrics
if y_pred is not None:
    print(f"Accuracy: {accuracy_score(yv, y_pred):.3f}")
    print("Confusion matrix:\n", confusion_matrix(yv, y_pred))
    print(classification_report(yv, y_pred))
    print(f"F1 @{threshold}:", round(f1_score(yv, y_pred), 5))
    f2 = fbeta_score(yv, y_pred, beta=2)
    print(f"F2 @{threshold}:", round(f2, 5))

# if threshold and probabilities are provided
if threshold is not None and proba is not None:
    y_pred = (proba >= threshold).astype(int)
    print("ROC-AUC:", round(roc_auc_score(yv, proba), 4))
    print("PR-AUC: ", round(average_precision_score(yv, proba), 5))
    p, r, thr = precision_recall_curve(yv, proba)
    f1s = 2 * p * r / (p + r + 1e-12)
    best = f1s[:-1].argmax()
    print(f"\nBest-threshold F1: {f1s[best]:.5f} at threshold {thr[best]:.5f}.")

    f2s = 5 * p * r / (4 * p + r + 1e-12)
    best2 = f2s[:-1].argmax()
    print(f"\nBest-threshold F2: {f2s[best2]:.5f} at threshold {thr[best2]:.5f}.")

Accuracy: 0.462
Confusion matrix:
 [[ 7604 10478]
 [  467  1805]]
              precision    recall  f1-score   support

           0       0.94      0.42      0.58     18082
           1       0.15      0.79      0.25      2272

    accuracy                           0.46     20354
   macro avg       0.54      0.61      0.41     20354
weighted avg       0.85      0.46      0.54     20354

F1 @0.65989: 0.24802
F2 @0.65989: 0.4223
ROC-AUC: 0.6668
PR-AUC:  0.20895

Best-threshold F1: 0.27633 at threshold 0.77373.

Best-threshold F2: 0.42358 at threshold 0.65565.


# Result

In [8]:
print(f"The model obtained an F2 score of {round(f2, 5)} at threshold {threshold}.")

The model obtained an F2 score of 0.4223 at threshold 0.65989.


# Weights

In [9]:
prep, model = clf.named_steps["prep"], clf.named_steps["model"]

def names_from(t, c):
    c = list(Xt.columns[c]) if isinstance(c, slice) else [c] if isinstance(c, str) else list(c)
    for s in reversed([s for _, s in t.steps] if hasattr(t, "steps") else [t]):
        for m in ("get_feature_names_out", "get_feature_names"):
            for a in ((c,), ()):
                try: return list(getattr(s, m)(*a))
                except Exception: pass
    return c  # passthrough / 1:1 transformer

names = [n for k, t, c in prep.transformers_ if not (isinstance(t, str) and t == "drop")
         for n in names_from(t, [Xt.columns[i] for i in c] if k == "remainder" and isinstance(t, str) else c)]

print(f"Model Intercept (Bias): {model.intercept_[0]:.4f}\n")

weights_df = pd.DataFrame({"Feature Name": names, "Weight (Coefficient)": model.coef_[0], "Percent Effect": np.exp(model.coef_[0])}).sort_values("Weight (Coefficient)", ascending=False, ignore_index=True)
display(weights_df)

Model Intercept (Bias): 0.5466



,Feature Name,Weight (Coefficient),Percent Effect
0,discharge_disposition_id_Discharged/transferred/referred to a psychiatric hospital of psychiatric distinct part unit of a hospital,1.810460,6.113256
1,diag_1_V58,1.632296,5.115606
2,discharge_disposition_id_Discharged/transferred to another rehab fac including rehab units of a hospital,1.425525,4.160040
3,discharge_disposition_id_Discharged/transferred to another type of inpatient care institution,1.077036,2.935963
4,discharge_disposition_id_Discharged/transferred to another short term hospital,0.690774,1.995258
...,...,...,...
149,diag_1_540,-0.852763,0.426235
150,discharge_disposition_id_Hospice / home,-0.864194,0.421391
151,diag_1_386,-1.060550,0.346265
152,diag_1_218,-1.188052,0.304814


In [10]:
weights_df.head(20)

,Feature Name,Weight (Coefficient),Percent Effect
0,discharge_disposition_id_Discharged/transferred/referred to a psychiatric hospital of psychiatric distinct part unit of a hospital,1.810460,6.113256
1,diag_1_V58,1.632296,5.115606
2,discharge_disposition_id_Discharged/transferred to another rehab fac including rehab units of a hospital,1.425525,4.160040
3,discharge_disposition_id_Discharged/transferred to another type of inpatient care institution,1.077036,2.935963
4,discharge_disposition_id_Discharged/transferred to another short term hospital,0.690774,1.995258
5,diag_1_572,0.690764,1.995240
6,discharge_disposition_id_infrequent_sklearn,0.665587,1.945632
7,diag_1_790,0.557990,1.747157
8,diag_1_157,0.548490,1.730638
9,diag_1_250.6,0.542834,1.720878


In [ ]:
weights_df.head(10)["Feature Name"].to_list()

['discharge_disposition_id_Discharged/transferred/referred to a psychiatric hospital of psychiatric distinct part unit of a hospital',
 'diag_1_V58',
 'discharge_disposition_id_Discharged/transferred to another rehab fac including rehab units of a hospital',
 'discharge_disposition_id_Discharged/transferred to another type of inpatient care institution',
 'discharge_disposition_id_Discharged/transferred to another short term hospital',
 'diag_1_572',
 'discharge_disposition_id_infrequent_sklearn',
 'diag_1_790',
 'diag_1_157',
 'diag_1_250.6']